# Olist 巴西电商数据集 — 数据探索

**数据库**：PostgreSQL · `analysis` 库 · `olist` 模式  
**数据规模**：9 张表 + 3 个视图，共 1,550,922 行  
**时间范围**：2016-09 至 2018-10

---
### 运行前提
右上角内核选择 `.venv` 对应的 Python 环境（项目内 `.venv` 文件夹）。

## 0. 环境准备与连接

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

# Windows 中文显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

DB = dict(host='127.0.0.1', port=5432, db='analysis', user='postgres', pwd='123456')
URL = 'postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}'.format(**DB)
engine = create_engine(URL, pool_pre_ping=True)

with engine.connect() as c:
    print('连接成功:', c.execute(text('SELECT current_database()')).scalar())

## 1. 数据概览
确认 9 张表的规模。

In [ ]:
sql = """
SELECT table_name AS 表名, table_type AS 类型
FROM information_schema.tables
WHERE table_schema = 'olist'
ORDER BY table_type, table_name
"""
pd.read_sql(sql, engine)

In [ ]:
sql = """
SELECT 'customers' AS 表, count(*) AS 行数 FROM olist.customers
UNION ALL SELECT 'geolocation',      count(*) FROM olist.geolocation
UNION ALL SELECT 'orders',           count(*) FROM olist.orders
UNION ALL SELECT 'order_items',      count(*) FROM olist.order_items
UNION ALL SELECT 'order_payments',   count(*) FROM olist.order_payments
UNION ALL SELECT 'order_reviews',    count(*) FROM olist.order_reviews
UNION ALL SELECT 'products',         count(*) FROM olist.products
UNION ALL SELECT 'sellers',          count(*) FROM olist.sellers
ORDER BY 1
"""
counts = pd.read_sql(sql, engine)
counts

## 2. 整体经营指标

In [ ]:
sql = """
SELECT count(*)                                          AS 订单数,
       count(DISTINCT customer_unique_id)                AS 去重客户数,
       round(sum(order_total)::numeric, 2)               AS 商品总额,
       round(avg(order_total)::numeric, 2)               AS 客单价,
       round(avg(review_score), 2)                       AS 平均评分,
       round(avg(EXTRACT(EPOCH FROM (delivered_at - purchase_at))/86400)::numeric, 1) AS 平均送达天数
FROM olist.v_orders_full
"""
kpi = pd.read_sql(sql, engine)
kpi.T.rename(columns={0: '数值'})

## 3. 销售趋势

In [ ]:
sql = """
SELECT to_char(purchase_month, 'YYYY-MM') AS 月份,
       count(*)                            AS 订单数,
       round(sum(order_total)::numeric, 2) AS 销售额
FROM olist.v_orders_full
GROUP BY 1 ORDER BY 1
"""
trend = pd.read_sql(sql, engine)
# 去掉数据不完整的头尾月份
trend = trend[(trend['订单数'] > 50)]
trend

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4.5))
ax1.bar(trend['月份'], trend['订单数'], color='#4A90D9', alpha=.85, label='订单数')
ax1.set_ylabel('订单数', color='#4A90D9')
ax1.tick_params(axis='x', rotation=60)

ax2 = ax1.twinx()
ax2.plot(trend['月份'], trend['销售额'], color='#E8833A', marker='o', lw=1.8, label='销售额')
ax2.set_ylabel('销售额 (R$)', color='#E8833A')

plt.title('Olist 月度订单量与销售额')
fig.tight_layout()
plt.show()

## 4. 地域分析

In [ ]:
sql = """
SELECT customer_state                      AS 州,
       count(*)                            AS 订单数,
       round(sum(order_total)::numeric, 2) AS 销售额,
       round(avg(order_total)::numeric, 2) AS 客单价,
       round(avg(review_score), 2)         AS 平均评分
FROM olist.v_orders_full
GROUP BY 1 ORDER BY 销售额 DESC
"""
states = pd.read_sql(sql, engine)
states.head(12)

In [ ]:
top = states.head(12)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(top['州'][::-1], top['销售额'][::-1], color='#4A90D9')
ax.set_xlabel('销售额 (R$)')
ax.set_title('各州销售额 Top 12')
for i, v in enumerate(top['销售额'][::-1]):
    ax.text(v, i, ' %.0f' % v, va='center', fontsize=8)
fig.tight_layout()
plt.show()

## 5. 品类分析

In [ ]:
sql = """
SELECT category_en                     AS 品类,
       sum(units_sold)                 AS 销量,
       round(sum(revenue)::numeric, 2) AS 销售额
FROM olist.v_product_sales
GROUP BY 1 ORDER BY 销售额 DESC NULLS LAST
"""
cats = pd.read_sql(sql, engine)
cats.head(15)

In [ ]:
top = cats.head(12)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top['品类'][::-1], top['销售额'][::-1], color='#5AA469')
ax.set_xlabel('销售额 (R$)')
ax.set_title('品类销售额 Top 12')
fig.tight_layout()
plt.show()

## 6. 评分分布

In [ ]:
sql = """
SELECT review_score AS 评分, count(*) AS 评论数
FROM olist.order_reviews GROUP BY 1 ORDER BY 1
"""
scores = pd.read_sql(sql, engine)
scores['占比%'] = (scores['评论数'] / scores['评论数'].sum() * 100).round(1)
scores

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#D9534F', '#E8833A', '#F0AD4E', '#7FB069', '#5AA469']
ax.bar(scores['评分'].astype(str), scores['评论数'], color=colors)
ax.set_xlabel('评分（星）')
ax.set_ylabel('评论数')
ax.set_title('评分分布')
for i, v in enumerate(scores['评论数']):
    ax.text(i, v, str(v), ha='center', va='bottom', fontsize=8)
fig.tight_layout()
plt.show()

## 7. 配送时效与满意度
验证一个常见假设：**送得越慢，评分越低**。

In [ ]:
sql = """
SELECT CASE
         WHEN delivered_at IS NULL THEN '未送达'
         WHEN EXTRACT(EPOCH FROM (delivered_at - purchase_at))/86400 <= 7  THEN '7天内'
         WHEN EXTRACT(EPOCH FROM (delivered_at - purchase_at))/86400 <= 14 THEN '8-14天'
         WHEN EXTRACT(EPOCH FROM (delivered_at - purchase_at))/86400 <= 21 THEN '15-21天'
         ELSE '21天以上' END AS 送达时长,
       count(*) AS 订单数,
       round(avg(review_score), 2) AS 平均评分
FROM olist.v_orders_full
GROUP BY 1
ORDER BY 平均评分 DESC NULLS LAST
"""
deliv = pd.read_sql(sql, engine)
deliv

In [ ]:
d = deliv[deliv['送达时长'] != '未送达']
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(d['送达时长'], d['平均评分'], color='#4A90D9')
ax.set_ylim(0, 5)
ax.set_ylabel('平均评分')
ax.set_title('送达时长 vs 平均评分')
for i, v in enumerate(d['平均评分']):
    ax.text(i, v + .05, str(v), ha='center', fontsize=9)
fig.tight_layout()
plt.show()

## 8. 支付方式

In [ ]:
sql = """
SELECT payment_type AS 支付方式,
       count(*) AS 笔数,
       round(sum(payment_value)::numeric, 2) AS 金额
FROM olist.order_payments GROUP BY 1 ORDER BY 金额 DESC
"""
pay = pd.read_sql(sql, engine)
pay['占比%'] = (pay['笔数'] / pay['笔数'].sum() * 100).round(1)
pay

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(pay['支付方式'], pay['金额'], color='#E8833A')
ax.set_ylabel('金额 (R$)')
ax.set_title('各支付方式金额')
for i, v in enumerate(pay['金额']):
    ax.text(i, v, '%.0f' % v, ha='center', va='bottom', fontsize=8)
fig.tight_layout()
plt.show()

## 9. 复购率

In [ ]:
sql = """
SELECT 下单次数, count(*) AS 客户数
FROM (SELECT customer_unique_id, count(*) AS 下单次数
      FROM olist.v_orders_full GROUP BY 1) t
GROUP BY 1 ORDER BY 1
"""
rep = pd.read_sql(sql, engine)
rep['占比%'] = (rep['客户数'] / rep['客户数'].sum() * 100).round(2)
print('总客户数:', rep['客户数'].sum())
print('复购率(下单>=2次): %.2f%%' % rep.loc[rep['下单次数'] >= 2, '占比%'].sum())
rep.head(10)

## 10. 小结与下一步

到这里你已经能从数据库直接读到数据并出图。可以继续做的方向：

1. **RFM 客户分层** —— 见 `sql/04_data_exploration.sql` 第 7.3 条
2. **迟到订单归因** —— 哪个州/哪个卖家延迟最严重
3. **品类关联分析** —— 同一个订单里常一起买的品类
4. **销售预测** —— 用月度序列做时间序列预测
5. **导出报告** —— `df.to_excel('结果.xlsx')` 或生成图表 PNG